# Section 3.2 Tutorial — The Multivariate Gaussian

This notebook is a guided tutorial for Section 3.2 of Bishop & Bishop, *Deep Learning: Foundations and Concepts*.

The goal is not just to memorize the multivariate Gaussian formula, but to build usable intuition:

- what the mean vector and covariance matrix do;
- why ellipses/ellipsoids appear;
- how marginal and conditional Gaussians work;
- how Gaussian Bayes updates work;
- how maximum likelihood estimates mean and covariance;
- why a single Gaussian is limited;
- why mixtures of Gaussians are much more flexible.

Where the section uses background not always emphasized in undergrad courses, this notebook adds short background notes: precision matrices, Schur complements, completing the square, matrix calculus, and latent variables/EM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

## 1. From 1D Gaussian to multivariate Gaussian

A one-dimensional Gaussian is:

$$
\mathcal{N}(x \mid \mu, \sigma^2)
=
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left\{-\frac{1}{2\sigma^2}(x-\mu)^2\right\}.
$$

The multivariate version replaces scalars with vectors and matrices:

| 1D idea | Multivariate idea |
|---|---|
| scalar value $x$ | vector $\mathbf{x}$ |
| scalar mean $\mu$ | mean vector $\boldsymbol{\mu}$ |
| variance $\sigma^2$ | covariance matrix $\boldsymbol{\Sigma}$ |
| squared distance $(x-\mu)^2/\sigma^2$ | Mahalanobis distance $(\mathbf{x}-\boldsymbol{\mu})^T\boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})$ |

The density is:

$$
\mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu},\boldsymbol{\Sigma})
=
\frac{1}{(2\pi)^{D/2}|\boldsymbol{\Sigma}|^{1/2}}
\exp\left\{
-\frac{1}{2}
(\mathbf{x}-\boldsymbol{\mu})^T
\boldsymbol{\Sigma}^{-1}
(\mathbf{x}-\boldsymbol{\mu})
\right\}.
$$

### Thinking model

A multivariate Gaussian is a **soft ellipsoid of probability mass**.

- $\boldsymbol{\mu}$ moves the ellipsoid.
- $\boldsymbol{\Sigma}$ shapes, stretches, and rotates it.
- $\boldsymbol{\Sigma}^{-1}$, called the **precision matrix**, measures distance in the geometry of the ellipsoid.
- $|\boldsymbol{\Sigma}|$ controls volume: larger covariance volume means lower peak density.

In [ ]:
def mvn_pdf(X, mu, Sigma):
    """Evaluate multivariate Gaussian density at X."""
    X = np.asarray(X)
    mu = np.asarray(mu)
    Sigma = np.asarray(Sigma)
    D = mu.shape[0]
    diff = X - mu
    inv = np.linalg.inv(Sigma)
    det = np.linalg.det(Sigma)
    exponent = -0.5 * np.einsum("...i,ij,...j->...", diff, inv, diff)
    norm = 1.0 / np.sqrt((2*np.pi)**D * det)
    return norm * np.exp(exponent)

def plot_gaussian_2d(mu, Sigma, title="2D Gaussian", samples_n=800):
    xs = np.linspace(mu[0]-4, mu[0]+4, 220)
    ys = np.linspace(mu[1]-4, mu[1]+4, 220)
    Xg, Yg = np.meshgrid(xs, ys)
    grid = np.dstack([Xg, Yg])
    Z = mvn_pdf(grid, mu, Sigma)
    samples = rng.multivariate_normal(mu, Sigma, size=samples_n)

    plt.figure(figsize=(7, 6))
    plt.contour(Xg, Yg, Z, levels=10)
    plt.scatter(samples[:, 0], samples[:, 1], s=8, alpha=0.35)
    plt.scatter([mu[0]], [mu[1]], marker="x", s=120)
    plt.axis("equal")
    plt.title(title)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.show()

mu = np.array([0.0, 0.0])
Sigma = np.array([[1.0, 0.8],
                  [0.8, 1.5]])
plot_gaussian_2d(mu, Sigma, "A correlated 2D Gaussian")

## 2. Geometry: covariance as shape

The covariance matrix must be symmetric and positive definite:

$$
\boldsymbol{\Sigma}=\boldsymbol{\Sigma}^T,
\qquad
\mathbf{v}^T\boldsymbol{\Sigma}\mathbf{v}>0
\quad \text{for every nonzero } \mathbf{v}.
$$

The positive-definite condition means every direction has positive variance.

### Background: eigenvectors and eigenvalues

For a covariance matrix,

$$
\boldsymbol{\Sigma}\mathbf{u}_i=\lambda_i\mathbf{u}_i.
$$

The eigenvectors $\mathbf{u}_i$ give the principal axes of the ellipse. The eigenvalues $\lambda_i$ give the variances along those axes.

- large eigenvalue: spread far in that direction;
- small eigenvalue: tightly concentrated in that direction;
- off-diagonal covariance: rotation/correlation.

In [ ]:
covariances = {
    "Independent, equal variance": np.array([[1.0, 0.0], [0.0, 1.0]]),
    "Independent, unequal variance": np.array([[2.0, 0.0], [0.0, 0.3]]),
    "Positive correlation": np.array([[1.0, 0.85], [0.85, 1.0]]),
    "Negative correlation": np.array([[1.0, -0.85], [-0.85, 1.0]]),
}

for title, S in covariances.items():
    print(title)
    print("eigenvalues:", np.linalg.eigvalsh(S))
    plot_gaussian_2d(np.array([0.0, 0.0]), S, title)

## 3. Mahalanobis distance

The Gaussian exponent uses:

$$
\Delta^2 =
(\mathbf{x}-\boldsymbol{\mu})^T
\boldsymbol{\Sigma}^{-1}
(\mathbf{x}-\boldsymbol{\mu}).
$$

This is called the **squared Mahalanobis distance**.

### Thinking model

Euclidean distance asks: *How far is $\mathbf{x}$ from $\boldsymbol{\mu}$ in ordinary space?*

Mahalanobis distance asks: *How surprising is $\mathbf{x}$, considering the directions where the data naturally varies?*

A point far along a high-variance direction may not be surprising. A point moderately far along a low-variance direction may be very surprising.

In [ ]:
mu = np.array([0.0, 0.0])
Sigma = np.array([[3.0, 1.2],
                  [1.2, 0.8]])
Lambda = np.linalg.inv(Sigma)

points = np.array([
    [2.0, 0.8],
    [0.5, 1.4],
    [2.0, -1.5],
])

for x in points:
    euclidean = np.linalg.norm(x - mu)
    mahal_sq = (x - mu) @ Lambda @ (x - mu)
    print(f"x={x}, Euclidean={euclidean:.3f}, Mahalanobis^2={mahal_sq:.3f}, density={mvn_pdf(x, mu, Sigma):.5f}")

plot_gaussian_2d(mu, Sigma, "Mahalanobis distance follows covariance geometry")

plt.figure(figsize=(6, 5))
samples = rng.multivariate_normal(mu, Sigma, size=500)
plt.scatter(samples[:, 0], samples[:, 1], s=8, alpha=0.25)
plt.scatter(points[:, 0], points[:, 1], s=80)
for i, x in enumerate(points):
    plt.text(x[0]+0.05, x[1]+0.05, f"p{i}")
plt.axis("equal")
plt.title("Same points over the sample cloud")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()

## 4. Moments

For a multivariate Gaussian:

$$
\mathbb{E}[\mathbf{x}] = \boldsymbol{\mu}
$$

and

$$
\mathrm{cov}[\mathbf{x}] =
\mathbb{E}\left[(\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^T\right]
=\boldsymbol{\Sigma}.
$$

Let's verify this by sampling.

In [ ]:
mu = np.array([1.0, -2.0])
Sigma = np.array([[2.0, 0.7],
                  [0.7, 1.0]])

X = rng.multivariate_normal(mu, Sigma, size=100_000)

print("True mean:")
print(mu)
print("\nSample mean:")
print(X.mean(axis=0))

print("\nTrue covariance:")
print(Sigma)
print("\nSample covariance:")
print(np.cov(X, rowvar=False, bias=True))

## 5. Limitations of a single Gaussian

A single Gaussian is mathematically convenient, but limited. It is always:

- unimodal: one main peak;
- elliptical in its equal-density contours;
- symmetric around its mean;
- fully described by mean and covariance.

This means a single Gaussian cannot naturally model data with multiple clumps, curved shapes, or heavy tails.

In [ ]:
# A data set with two clumps
N = 600
X1 = rng.multivariate_normal([-2, 0], [[0.35, 0.0], [0.0, 0.35]], size=N//2)
X2 = rng.multivariate_normal([2, 1], [[0.45, 0.2], [0.2, 0.45]], size=N//2)
X = np.vstack([X1, X2])

mu_hat = X.mean(axis=0)
Sigma_hat = np.cov(X, rowvar=False, bias=True)

plt.figure(figsize=(7, 6))
plt.scatter(X[:, 0], X[:, 1], s=10, alpha=0.4)

xs = np.linspace(-4, 4, 220)
ys = np.linspace(-3, 4, 220)
Xg, Yg = np.meshgrid(xs, ys)
grid = np.dstack([Xg, Yg])
Z = mvn_pdf(grid, mu_hat, Sigma_hat)
plt.contour(Xg, Yg, Z, levels=8)
plt.axis("equal")
plt.title("A single Gaussian tries to cover both clumps")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()

print("Fitted mean:", mu_hat)
print("Fitted covariance:\n", Sigma_hat)

## 6. Partitioning a Gaussian

Suppose the vector $\mathbf{x}$ is split into two parts:

$$
\mathbf{x}=\begin{bmatrix}\mathbf{x}_a\\\mathbf{x}_b\end{bmatrix},
\qquad
\boldsymbol{\mu}=\begin{bmatrix}\boldsymbol{\mu}_a\\\boldsymbol{\mu}_b\end{bmatrix},
\qquad
\boldsymbol{\Sigma}=\begin{bmatrix}
\boldsymbol{\Sigma}_{aa} & \boldsymbol{\Sigma}_{ab}\\
\boldsymbol{\Sigma}_{ba} & \boldsymbol{\Sigma}_{bb}
\end{bmatrix}.
$$

This notation is the basis for marginal and conditional Gaussians.

### Marginal distribution

If the joint vector is Gaussian, then:

$$
\mathbf{x}_a \sim \mathcal{N}(\boldsymbol{\mu}_a,\boldsymbol{\Sigma}_{aa})
$$

and

$$
\mathbf{x}_b \sim \mathcal{N}(\boldsymbol{\mu}_b,\boldsymbol{\Sigma}_{bb}).
$$

### Thinking model

Marginalizing a Gaussian means: **ignore some coordinates**. The remaining coordinates are still Gaussian.

In [ ]:
# 3D Gaussian. We will look only at dimensions 0 and 1.
mu = np.array([1.0, -1.0, 2.0])
Sigma = np.array([
    [2.0, 0.8, 0.4],
    [0.8, 1.5, -0.3],
    [0.4, -0.3, 1.0]
])

X3 = rng.multivariate_normal(mu, Sigma, size=50_000)

print("Full mean estimate:", X3.mean(axis=0))
print("Full covariance estimate:\n", np.cov(X3, rowvar=False, bias=True))

Xa = X3[:, :2]
print("\nMarginal mean for first two coordinates:", Xa.mean(axis=0))
print("Should be:", mu[:2])

print("\nMarginal covariance for first two coordinates:\n", np.cov(Xa, rowvar=False, bias=True))
print("Should be:\n", Sigma[:2, :2])

## 7. Conditional Gaussian

For a joint Gaussian split into $\mathbf{x}_a$ and $\mathbf{x}_b$,

$$
p(\mathbf{x}_a\mid\mathbf{x}_b)
$$

is also Gaussian:

$$
\mathbf{x}_a\mid\mathbf{x}_b
\sim
\mathcal{N}(\boldsymbol{\mu}_{a|b},\boldsymbol{\Sigma}_{a|b}).
$$

Using covariance blocks:

$$
\boldsymbol{\mu}_{a|b}
=
\boldsymbol{\mu}_a
+
\boldsymbol{\Sigma}_{ab}
\boldsymbol{\Sigma}_{bb}^{-1}
(\mathbf{x}_b-\boldsymbol{\mu}_b)
$$

and

$$
\boldsymbol{\Sigma}_{a|b}
=
\boldsymbol{\Sigma}_{aa}
-
\boldsymbol{\Sigma}_{ab}
\boldsymbol{\Sigma}_{bb}^{-1}
\boldsymbol{\Sigma}_{ba}.
$$

### Thinking model

Conditioning says: **I observed part of the vector. How should my belief about the unobserved part change?**

If $\mathbf{x}_a$ and $\mathbf{x}_b$ are correlated, observing $\mathbf{x}_b$ shifts the mean of $\mathbf{x}_a$.

The conditional covariance is smaller than or equal to the marginal covariance because observing $\mathbf{x}_b$ gives information.

In [ ]:
def conditional_gaussian(mu, Sigma, idx_a, idx_b, x_b):
    """Return mean and covariance of x_a | x_b for a joint Gaussian N(mu, Sigma)."""
    idx_a = np.array(idx_a)
    idx_b = np.array(idx_b)
    mu_a = mu[idx_a]
    mu_b = mu[idx_b]
    S_aa = Sigma[np.ix_(idx_a, idx_a)]
    S_ab = Sigma[np.ix_(idx_a, idx_b)]
    S_ba = Sigma[np.ix_(idx_b, idx_a)]
    S_bb = Sigma[np.ix_(idx_b, idx_b)]

    K = S_ab @ np.linalg.inv(S_bb)
    cond_mu = mu_a + K @ (x_b - mu_b)
    cond_S = S_aa - K @ S_ba
    return cond_mu, cond_S

mu2 = np.array([0.0, 0.0])
Sigma2 = np.array([[1.0, 0.85],
                   [0.85, 1.0]])

for observed_x2 in [-1.5, 0.0, 1.5]:
    cmu, cS = conditional_gaussian(mu2, Sigma2, idx_a=[0], idx_b=[1], x_b=np.array([observed_x2]))
    print(f"x2={observed_x2:+.1f} -> x1|x2 mean={cmu[0]:+.3f}, variance={cS[0,0]:.3f}")

plot_gaussian_2d(mu2, Sigma2, "Positive correlation: observing x2 shifts belief about x1")

### Visualizing conditional slices

For a 2D Gaussian, conditioning on $x_2=c$ gives a 1D Gaussian over $x_1$.

If the variables are positively correlated, large $x_2$ implies large expected $x_1$.

In [ ]:
def normal_pdf_1d(x, mu, var):
    return (1 / np.sqrt(2*np.pi*var)) * np.exp(-0.5 * (x-mu)**2 / var)

x1_grid = np.linspace(-4, 4, 400)
observed_values = [-1.5, 0.0, 1.5]

plt.figure(figsize=(8, 5))
for x2 in observed_values:
    cmu, cS = conditional_gaussian(mu2, Sigma2, [0], [1], np.array([x2]))
    density = normal_pdf_1d(x1_grid, cmu[0], cS[0,0])
    plt.plot(x1_grid, density, label=f"$x_2={x2}$, mean={cmu[0]:.2f}")

plt.title("Conditional distributions $p(x_1 \\mid x_2)$")
plt.xlabel("$x_1$")
plt.ylabel("density")
plt.legend()
plt.show()

## 8. Background not always covered: completing the square

The conditional Gaussian formula can be derived by looking at the exponent and completing the square.

For a scalar quadratic:

$$
ax^2 - 2bx + c
=
a\left(x-\frac{b}{a}\right)^2 + \text{constant}.
$$

For vectors:

$$
\mathbf{x}^T\mathbf{A}\mathbf{x}
-2\mathbf{b}^T\mathbf{x}+c
=
(\mathbf{x}-\mathbf{A}^{-1}\mathbf{b})^T
\mathbf{A}
(\mathbf{x}-\mathbf{A}^{-1}\mathbf{b})
+\text{constant}.
$$

This tells us:

- precision matrix: $\mathbf{A}$;
- covariance matrix: $\mathbf{A}^{-1}$;
- mean: $\mathbf{A}^{-1}\mathbf{b}$.

This is why many Gaussian conditional formulas are cleaner in terms of the **precision matrix**.

## 9. Background not always covered: block matrices and Schur complements

The expression

$$
\boldsymbol{\Sigma}_{aa}
-
\boldsymbol{\Sigma}_{ab}
\boldsymbol{\Sigma}_{bb}^{-1}
\boldsymbol{\Sigma}_{ba}
$$

is a **Schur complement**.

### Why it matters

It appears when you ask: **How much variance remains in $\mathbf{x}_a$ after accounting for $\mathbf{x}_b$?**

So the conditional covariance is the leftover uncertainty after using $\mathbf{x}_b$ to predict $\mathbf{x}_a$ linearly.

In [ ]:
# Verify conditional covariance is smaller than marginal variance in a scalar example.
rho_values = np.linspace(-0.95, 0.95, 11)

print("rho    marginal Var(x1)    conditional Var(x1|x2)")
for rho in rho_values:
    S = np.array([[1.0, rho],
                  [rho, 1.0]])
    _, cS = conditional_gaussian(np.zeros(2), S, [0], [1], np.array([0.0]))
    print(f"{rho:+.2f}       {1.0:.3f}              {cS[0,0]:.3f}")

For two standardized variables with correlation $\rho$,

$$
\mathrm{Var}(x_1\mid x_2)=1-\rho^2.
$$

So if $|\rho|$ is large, $x_2$ tells us a lot about $x_1$.

## 10. Linear-Gaussian Bayes theorem

A very important setup is:

$$
p(\mathbf{x})=\mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu},\boldsymbol{\Lambda}^{-1})
$$

and

$$
p(\mathbf{y}\mid\mathbf{x})
=\mathcal{N}(\mathbf{y}\mid\mathbf{A}\mathbf{x}+\mathbf{b},\mathbf{L}^{-1}).
$$

Here:

- $p(\mathbf{x})$ is a Gaussian prior;
- $p(\mathbf{y}\mid\mathbf{x})$ is a Gaussian likelihood;
- $p(\mathbf{x}\mid\mathbf{y})$ is the Gaussian posterior.

The posterior covariance is:

$$
\boldsymbol{\Sigma}=(\boldsymbol{\Lambda}+\mathbf{A}^T\mathbf{L}\mathbf{A})^{-1}
$$

and the posterior mean is:

$$
\mathbb{E}[\mathbf{x}\mid\mathbf{y}]
=
\boldsymbol{\Sigma}
\left[
\mathbf{A}^T\mathbf{L}(\mathbf{y}-\mathbf{b})
+
\boldsymbol{\Lambda}\boldsymbol{\mu}
\right].
$$

### Thinking model

**Prior precision + data precision = posterior precision.**

In [ ]:
# Scalar version: infer unknown x from noisy observation y = x + noise.
prior_mu = 0.0
prior_var = 4.0      # uncertain prior
noise_var = 1.0      # observation noise
y = 3.0              # observed value

prior_precision = 1 / prior_var
noise_precision = 1 / noise_var

posterior_var = 1 / (prior_precision + noise_precision)
posterior_mu = posterior_var * (noise_precision * y + prior_precision * prior_mu)

print("prior mean, var:", prior_mu, prior_var)
print("observed y:", y)
print("posterior mean, var:", posterior_mu, posterior_var)

grid1 = np.linspace(-6, 6, 500)
prior = normal_pdf_1d(grid1, prior_mu, prior_var)
likelihood_as_fn_of_x = normal_pdf_1d(y, grid1, noise_var)  # p(y|x), viewed as a function of x
posterior = normal_pdf_1d(grid1, posterior_mu, posterior_var)

plt.figure(figsize=(8, 5))
plt.plot(grid1, prior, label="prior $p(x)$")
plt.plot(grid1, likelihood_as_fn_of_x, label="likelihood $p(y|x)$ as function of $x$")
plt.plot(grid1, posterior, label="posterior $p(x|y)$")
plt.title("Scalar Gaussian Bayes update")
plt.xlabel("$x$")
plt.ylabel("density up to scale")
plt.legend()
plt.show()

## 11. Maximum likelihood for a multivariate Gaussian

Given data points $\mathbf{x}_1,\ldots,\mathbf{x}_N$ assumed IID from

$$
\mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu},\boldsymbol{\Sigma}),
$$

the maximum likelihood estimates are:

$$
\boldsymbol{\mu}_{ML}
=
\frac{1}{N}\sum_{n=1}^N \mathbf{x}_n
$$

and

$$
\boldsymbol{\Sigma}_{ML}
=
\frac{1}{N}
\sum_{n=1}^N
(\mathbf{x}_n-\boldsymbol{\mu}_{ML})
(\mathbf{x}_n-\boldsymbol{\mu}_{ML})^T.
$$

### Background not always covered: matrix calculus

The log likelihood contains:

$$
-\frac{N}{2}\ln|\boldsymbol{\Sigma}|
-
\frac{1}{2}
\sum_n
(\mathbf{x}_n-\boldsymbol{\mu})^T
\boldsymbol{\Sigma}^{-1}
(\mathbf{x}_n-\boldsymbol{\mu}).
$$

To derive the covariance estimate rigorously, you need matrix derivative identities involving:

- derivative of $\ln|\boldsymbol{\Sigma}|$;
- derivative of quadratic forms;
- symmetry/positive-definiteness constraints.

For practical ML work, the key result is: MLE covariance is the average outer product of centered data.

In [ ]:
true_mu = np.array([1.0, 2.0])
true_Sigma = np.array([[2.0, 0.9],
                       [0.9, 1.0]])

Xmle = rng.multivariate_normal(true_mu, true_Sigma, size=500)

mu_ml = Xmle.mean(axis=0)
Sigma_ml = (Xmle - mu_ml).T @ (Xmle - mu_ml) / len(Xmle)

print("True mean:", true_mu)
print("ML mean:", mu_ml)

print("\nTrue covariance:\n", true_Sigma)
print("\nML covariance:\n", Sigma_ml)

### Bias of the covariance MLE

The MLE covariance uses division by $N$.

Its expectation is:

$$
\mathbb{E}[\boldsymbol{\Sigma}_{ML}]
=
\frac{N-1}{N}\boldsymbol{\Sigma}.
$$

So it is biased low.

The usual unbiased covariance estimate divides by $N-1$:

$$
\hat{\boldsymbol{\Sigma}}
=
\frac{1}{N-1}
\sum_{n=1}^N
(\mathbf{x}_n-\boldsymbol{\mu}_{ML})
(\mathbf{x}_n-\boldsymbol{\mu}_{ML})^T.
$$

For large $N$, the difference is small. For small $N$, it matters.

In [ ]:
# Monte Carlo demonstration of bias.
true_mu = np.array([0.0, 0.0])
true_Sigma = np.array([[1.0, 0.5],
                       [0.5, 2.0]])

N_small = 5
trials = 20_000
cov_ml_sum = np.zeros((2, 2))
cov_unbiased_sum = np.zeros((2, 2))

for _ in range(trials):
    Xtrial = rng.multivariate_normal(true_mu, true_Sigma, size=N_small)
    mu_hat = Xtrial.mean(axis=0)
    centered = Xtrial - mu_hat
    cov_ml_sum += centered.T @ centered / N_small
    cov_unbiased_sum += centered.T @ centered / (N_small - 1)

print("True covariance:\n", true_Sigma)
print("\nAverage ML covariance over trials:\n", cov_ml_sum / trials)
print("\nAverage unbiased covariance over trials:\n", cov_unbiased_sum / trials)
print("\nExpected ML factor:", (N_small-1)/N_small)

## 12. Sequential estimation of the mean

Batch estimate:

$$
\boldsymbol{\mu}^{(N)}_{ML}
=
\frac{1}{N}\sum_{n=1}^N \mathbf{x}_n.
$$

Sequential update:

$$
\boldsymbol{\mu}^{(N)}_{ML}
=
\boldsymbol{\mu}^{(N-1)}_{ML}
+
\frac{1}{N}
\left(
\mathbf{x}_N-\boldsymbol{\mu}^{(N-1)}_{ML}
\right).
$$

### Thinking model

The new point sends an **error signal**:

$$
\mathbf{x}_N-\boldsymbol{\mu}^{(N-1)}_{ML}.
$$

The learning rate is $1/N$. Early points move the mean a lot. Later points move it only a little.

In [ ]:
Xseq = rng.multivariate_normal([2, -1], [[1, 0.3], [0.3, 1]], size=200)

running_mu = np.zeros(2)
path = []

for n, x in enumerate(Xseq, start=1):
    running_mu = running_mu + (1/n) * (x - running_mu)
    path.append(running_mu.copy())

path = np.array(path)

print("Batch mean:", Xseq.mean(axis=0))
print("Sequential mean:", running_mu)

plt.figure(figsize=(7, 6))
plt.scatter(Xseq[:, 0], Xseq[:, 1], s=10, alpha=0.25, label="data")
plt.plot(path[:, 0], path[:, 1], marker=".", markersize=2, label="running mean path")
plt.scatter([Xseq.mean(axis=0)[0]], [Xseq.mean(axis=0)[1]], marker="x", s=120, label="final mean")
plt.axis("equal")
plt.title("Sequential mean estimate")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend()
plt.show()

## 13. Mixtures of Gaussians

A single Gaussian is one ellipsoid.

A mixture of Gaussians is a weighted sum of ellipsoids:

$$
p(\mathbf{x})
=
\sum_{k=1}^K
\pi_k
\mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k)
$$

where:

$$
\sum_{k=1}^K \pi_k=1,
\qquad
\pi_k\ge 0.
$$

### Thinking model

A Gaussian mixture says: **Each data point probably came from one of several hidden Gaussian sources.**

The hidden source is a **latent variable**.

In [ ]:
def gmm_pdf(X, pis, mus, Sigmas):
    total = np.zeros(X.shape[:-1])
    for pi, mu, S in zip(pis, mus, Sigmas):
        total += pi * mvn_pdf(X, mu, S)
    return total

pis = np.array([0.45, 0.55])
mus = [np.array([-2, 0]), np.array([2, 1])]
Sigmas = [
    np.array([[0.35, 0.0], [0.0, 0.35]]),
    np.array([[0.45, 0.2], [0.2, 0.45]])
]

xs = np.linspace(-4, 4, 220)
ys = np.linspace(-3, 4, 220)
Xg, Yg = np.meshgrid(xs, ys)
grid = np.dstack([Xg, Yg])
Z = gmm_pdf(grid, pis, mus, Sigmas)

plt.figure(figsize=(7, 6))
plt.contour(Xg, Yg, Z, levels=12)
plt.scatter(X[:, 0], X[:, 1], s=10, alpha=0.25)
plt.axis("equal")
plt.title("Two-component Gaussian mixture")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()

## 14. Background not always covered: latent variables and EM

A mixture model has a hidden discrete variable $z$:

$$
z\in\{1,\ldots,K\}.
$$

It chooses which Gaussian component generated the point.

The model is:

$$
p(z=k)=\pi_k
$$

and

$$
p(\mathbf{x}\mid z=k)
=\mathcal{N}(\mathbf{x}\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}_k).
$$

Then:

$$
p(\mathbf{x})
=
\sum_{k=1}^K p(z=k)p(\mathbf{x}\mid z=k).
$$

Training a Gaussian mixture is usually done with the **EM algorithm**:

1. **E-step:** estimate soft assignments, called responsibilities:
   $$
   r_{nk}=p(z_n=k\mid\mathbf{x}_n).
   $$
2. **M-step:** refit each Gaussian using weighted data points.

We will study EM later, but this small implementation gives the core idea.

In [ ]:
def fit_gmm_em(X, K=2, steps=25, seed=0):
    rng_local = np.random.default_rng(seed)
    N, D = X.shape

    # Initialize means by picking random data points.
    mus = X[rng_local.choice(N, K, replace=False)].copy()
    Sigmas = np.array([np.cov(X, rowvar=False) + 1e-3*np.eye(D) for _ in range(K)])
    pis = np.ones(K) / K

    log_likelihoods = []

    for step in range(steps):
        # E-step
        weighted = np.zeros((N, K))
        for k in range(K):
            weighted[:, k] = pis[k] * mvn_pdf(X, mus[k], Sigmas[k])
        normalizer = weighted.sum(axis=1, keepdims=True)
        responsibilities = weighted / normalizer

        # M-step
        Nk = responsibilities.sum(axis=0)
        pis = Nk / N
        for k in range(K):
            mus[k] = (responsibilities[:, [k]] * X).sum(axis=0) / Nk[k]
            centered = X - mus[k]
            Sigmas[k] = (responsibilities[:, [k]] * centered).T @ centered / Nk[k]
            Sigmas[k] += 1e-6 * np.eye(D)

        ll = np.sum(np.log(normalizer[:, 0]))
        log_likelihoods.append(ll)

    return pis, mus, Sigmas, np.array(log_likelihoods), responsibilities

pis_hat, mus_hat, Sigmas_hat, lls, R = fit_gmm_em(X, K=2, steps=30, seed=3)

print("Estimated mixture weights:", pis_hat)
print("Estimated means:\n", mus_hat)
print("Estimated covariances:\n", Sigmas_hat)

plt.figure(figsize=(7, 4))
plt.plot(lls, marker="o")
plt.title("EM increases the log likelihood")
plt.xlabel("EM step")
plt.ylabel("log likelihood")
plt.show()

Z_hat = gmm_pdf(grid, pis_hat, mus_hat, Sigmas_hat)

plt.figure(figsize=(7, 6))
plt.scatter(X[:, 0], X[:, 1], c=R[:, 0], s=12, alpha=0.6)
plt.contour(Xg, Yg, Z_hat, levels=12)
plt.axis("equal")
plt.title("Fitted Gaussian mixture with soft assignments")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()

## 15. What to remember

### Core facts

1. A multivariate Gaussian is determined by a mean vector and covariance matrix.

$$
\mathbf{x}\sim\mathcal{N}(\boldsymbol{\mu},\boldsymbol{\Sigma})
$$

2. Its density is shaped by Mahalanobis distance:

$$
(\mathbf{x}-\boldsymbol{\mu})^T
\boldsymbol{\Sigma}^{-1}
(\mathbf{x}-\boldsymbol{\mu}).
$$

3. Marginals of a Gaussian are Gaussian.
4. Conditionals of a Gaussian are Gaussian.
5. Linear-Gaussian Bayes updates are analytically tractable.
6. MLE estimates are sample mean and sample covariance.
7. A single Gaussian is limited; mixtures can model multiple clumps.

### Mental compression

- **Covariance:** geometry of uncertainty.
- **Precision:** inverse uncertainty; useful for conditioning and Bayesian updates.
- **Marginalization:** forget coordinates.
- **Conditioning:** observe coordinates and shrink/shift belief.
- **Mixture:** several Gaussian explanations combined.

## 16. Practice problems

Try these before looking at the answers.

### Problem 1

Let

$$
\boldsymbol{\mu}=\begin{bmatrix}0\\0\end{bmatrix},
\qquad
\boldsymbol{\Sigma}=\begin{bmatrix}1&0.6\\0.6&1\end{bmatrix}.
$$

Compute the distribution of $x_1\mid x_2=2$.

### Problem 2

Generate 1000 samples from a 2D Gaussian with mean $[3,-1]$ and covariance

$$
\begin{bmatrix}4&-1\\-1&1\end{bmatrix}.
$$

Estimate the mean and covariance.

### Problem 3

Explain why a single Gaussian is a bad model for two separated clusters.

### Problem 4

In a Gaussian Bayes update, why do precisions add?

In [ ]:
# Scratch space for practice problems.

## 17. Answers

### Answer 1

Here:

$$
\mu_1=0,\quad \mu_2=0,\quad
\Sigma_{11}=1,\quad \Sigma_{12}=0.6,\quad \Sigma_{22}=1.
$$

So:

$$
\mathbb{E}[x_1\mid x_2=2]
=0+0.6(1)^{-1}(2-0)=1.2.
$$

and

$$
\mathrm{Var}(x_1\mid x_2)=1-0.6^2=0.64.
$$

Therefore:

$$
x_1\mid x_2=2\sim\mathcal{N}(1.2,0.64).
$$

### Answer 2

Use `rng.multivariate_normal`, then `X.mean(axis=0)` and `np.cov(X, rowvar=False)`.

### Answer 3

A single Gaussian has one mean and elliptical contours. For two separated clusters, it often places high density between the clusters, exactly where few or no data points occur.

### Answer 4

Precision is inverse variance. In a linear-Gaussian update, the posterior combines the prior information and observation information. Independent Gaussian information sources contribute additively in the quadratic exponent, so their precision terms add.

In [ ]:
# Check Answer 1 with the helper function.
mu = np.array([0.0, 0.0])
Sigma = np.array([[1.0, 0.6],
                  [0.6, 1.0]])

conditional_gaussian(mu, Sigma, idx_a=[0], idx_b=[1], x_b=np.array([2.0]))